# Mini-Projet NLP : Analyse Automatique de Sentiments
**Module :** IA et ses Applications • **Projet N° 05**  
**Binôme 5 :** Hamza BAADI & Nadya OUFDIME  
**Livrable :** Notebook d'expérimentation, prétraitement, modélisation et analyse d'erreurs.

## 1. Importation des bibliothèques nécessaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib

## 2. Chargement et Exploration du Corpus d'Avis (Section 4.B)
Étude de l'équilibre des classes, de la longueur des avis et de la qualité.

In [ ]:
from train_model import get_sentiment_dataset

df = get_sentiment_dataset()
print(f"Taille du dataset : {len(df)} commentaires")
print(df['label'].value_counts())
df.head()

## 3. Découpage Train / Test (Section 4.C)
**Important :** Découpage strict avant ajustement du TF-IDF pour éviter toute fuite de données (*data leakage*).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
)
print(f"Ensemble d'entraînement : {len(X_train)} | Ensemble de test : {len(X_test)}")

## 4. Modélisation et Comparaison (Section 4.D & 4.E)
- **Baseline :** DummyClassifier (Most Frequent)
- **Modèle 1 :** Pipeline TF-IDF + Régression Logistique
- **Modèle 2 :** Pipeline TF-IDF + Multinomial Naive Bayes

In [ ]:
# Baseline
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

# Pipeline TF-IDF + Régression Logistique
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, lowercase=True)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
pred_lr = pipe_lr.predict(X_test)

# Pipeline TF-IDF + Naive Bayes
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, lowercase=True)),
    ('clf', MultinomialNB(alpha=0.5))
])
pipe_nb.fit(X_train, y_train)
pred_nb = pipe_nb.predict(X_test)

print("=== Régression Logistique ===")
print(classification_report(y_test, pred_lr, target_names=['Négatif', 'Positif']))

## 5. Matrice de Confusion et Analyse des 5 Erreurs Concrètes (Section 4.F)

In [ ]:
cm = confusion_matrix(y_test, pred_lr)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Négatif', 'Positif'], yticklabels=['Négatif', 'Positif'])
plt.title('Matrice de Confusion (Régression Logistique)')
plt.xlabel('Prédiction')
plt.ylabel('Réel')
plt.show()

## 6. Sauvegarde du Pipeline Final pour Streamlit

In [ ]:
joblib.dump(pipe_lr, 'models/sentiment_pipeline.joblib')
print("Modèle et vectoriseur TF-IDF sauvegardés dans models/sentiment_pipeline.joblib !")